## Limpando epoched data

In [ ]:
import pathlib
import matplotlib

import mne
import mne_bids

matplotlib.use('Qt5Agg')
mne.set_log_level('warning') # Mostre somente mensagens do tipo “warning” ou mais importantes (error, critical).Oculte mensagens informativas e de debug.

'''
O MNE tem um sistema próprio de log (mensagens informativas, avisos, alertas, erros), o qual imprime MUITA coisa na tela.
Durante pré-processamento (filter, epoching, ICA...), são dezenas de logs.

Se você quer:
- menos poluição visual
- só ver avisos importantes
- logs mais limpos no Jupyter

então usar 'warning' é perfeito, porque queremos memos mensagens de 'deu certo' na tela


mne.set_log_level('info')     # padrão, bastante verboso
mne.set_log_level('warning')  # mostrar avisos e todas as mensagens que têm nível mais grave que aviso.
mne.set_log_level('error')    # só erros
mne.set_log_level('critical') # quase nada aparece
mne.set_log_level('debug')    # extremamente detalhado

Da menos grave → mais grave:

- DEBUG (detalhes internos, muito verboso)
- INFO (mensagens normais, informativas)
- WARNING (avisos importantes)
- ERROR (um erro ocorreu)
- CRITICAL (erro que pode interromper tudo)
'''

In [ ]:
epochs = mne.read_epochs(pathlib.Path('out_data') / 'epochs_epo.fif')
epochs

In [ ]:
# Aplicando minha correção com baseline. Provavelmente já está aplicado, mas vamos garantir
epochs.apply_baseline((None, 0))

In [ ]:
epochs.plot()

## Rejeitar artefatos com base na amplitude do sinal do canal

O MNE pode descartar algumas das nossas epochs, baseado na amplitude do sinal. Isso é chamado de *specifying a rejection criteria*

In [ ]:
# rejeita epochs com picos altos (artefatos fortes).
reject_criteria = dict(mag=3000e-15,     # 3000 fT
                       grad=3000e-13,    # 3000 fT/cm
                       eeg=150e-6,       # 150 µV
                       eog=200e-6)       # 200 µV

# Para EEG: se o sinal passar de 150 µV, é provável que tenha piscada, movimento, tensão muscular etc.


# Se algum canal tiver amplitude quase zero (epochs muito baixas, indicando canal morto) → marca como “flat” e rejeita
flat_criteria = dict(mag=1e-15,          # 1 fT
                     grad=1e-13,         # 1 fT/cm
                     eeg=1e-6)           # 1 µV

### O “zero” no gráfico é o quê?

É simplesmente:

*“Mesmo nível da referência.”*

Se o canal está acima desse nível → amplitude positiva
Se está abaixo → amplitude negativa

Não tem nenhum significado físico absoluto.
É só a diferença instantânea entre o eletrodo e a referência

Se você muda a referência (ex: de mastoide para média dos canais), o “zero” muda completamente, porque a base de comparação muda.
Mas a atividade cerebral é a mesma — apenas a escala e o equilíbrio mudam.

---


## Como saber se a amplitude está “correta”?

### Critérios práticos para EEG bom:

* **Faixa geral**: entre **5 e 100 µV**
* Canal limpo: **5–20 µV**
* Artefato de piscar: até **200 µV**
* Nada de flat (0 µV)
* Nada de explodido (> 500–1000 µV)

---

In [ ]:
epochs.drop_bad(reject=reject_criteria, flat=flat_criteria)

# Perceba a quantidade de eventos diminuir

In [ ]:
epochs.drop_log

In [ ]:
# Visualizar de forma gráfica e organizada quais epochs foram rejeitadas, e por qual motivo.

epochs.plot_drop_log() # Obviamente tenho que rodar 'epochs.drop_bad(reject=reject_criteria, flat=flat_criteria)' para obter algo

Um número considerável de epochs foi removido devido à quantidade de artefatos de EOG

In [ ]:
epochs['Visual'].plot_image() # heatmap interativo de sinal

In [ ]:
epochs.plot_sensors(ch_type='eeg') # Tem algum sensor 'saindo' do mapa, ou seja, na parte de trás da cabeça?

# Têm mais de um! O principal (mais distante) é o EEG 060

In [ ]:
epochs['Visual'].plot_image(picks='EEG 060')


'''
✓ A amplitude é baixa e consistente

Dá para ver cores fracas, majoritariamente entre ±10 μV, o que é absolutamente normal para EEG limpo.

— Nada perto de 100–200 μV (piscadas).
— Nada perto de 50–100 μV (EMG pesado).
— Nada flat (canal morto)


✓ O ERP (traço preto embaixo) está suave

Um canal ruim gera um ERP “serrilhado”, denteado, oscilando demais.
O seu ERP:

- é suave
- tem forma fisiológica
- tem intervalo de confiança estreito (sombreamento cinza)
→ isso é sinal de canal saudável
'''


## Por que o gráfico parece “baixo”, tipo ±8 µV?

Porque você está olhando um **canal fisiológico estável**.

### EEG limpo pode perfeitamente ficar entre:

**±5 µV e ±20 µV**

Isso é absolutamente normal.


Se o seu canal **não passa de 8 µV**, isso indica:

* eletrodo bem conectado
* baixa interferência muscular
* ausência de artefatos de piscar
* ruído ambiental baixo

Ou seja, **bom sinal.**

---


Primeiro, comece novamente com os dados brutos e aplique um filtro passa-alta de 1,0 Hz, o que é vantajoso para o desempenho da ICA.

In [ ]:
# Novamente, preciso recarregar o raw

'''
O ICA deve, preferencialmente, ser ajustado em dado contínuo, e não em Epochs, porque:

- Epocar corta o sinal e remove pedaços (saltos na borda → ICA odeia isso),
- Epocar já remove trechos com ruído,
- ICA precisa ver todo o fluxo temporal dos artefatos (piscar, movimento, etc.).

Entretanto, estaremos trabalhando com o ICA em epochs para facilitação de análise

'''

bids_root = pathlib.Path('out_data/sample_BIDS')


'''
LEMBRANDO 

O MNE sempre lê o arquivo como ele está:

✔ raw original
✔ sem filtros
✔ sem ICA
✔ sem SSP
✔ sem epochs
✔ sem baseline
✔ sem nenhuma modificação anterior

Ou seja:

Você está pegando o dado bruto “do zero” do subject
01, do jeito que veio do laboratório.
'''

bids_path = mne_bids.BIDSPath(subject='01',
                              session='01',
                              task='audiovisual',
                              run='01',
                              datatype='meg',
                              root=bids_root)



raw = mne_bids.read_raw_bids(bids_path)
raw.load_data()
raw.filter(l_freq=1, h_freq=40)  # High-pass with 1. Hz cut-off is recommended for ICA

Leia nossas epochs e extraia aquelas que foram mantidas (todas no nosso caso, porque não aplicamos nenhum procedimento de rejeição antes de salvar as épocas no notebook 3; mas isso poderia ser diferente em um cenário real, e você quer calcular o ICA no mesmo conjunto de épocas que está realmente alimentando na sua análise!).

In [ ]:
# Filtrei meu dado bruto para o ICA e agora quero dividir ele em epochs (como fiz no notebook 3)
# Só que, posso ter excluído algumas manualmente através da janelinha interativa e, quero, manter elas 
epochs = mne.read_epochs(pathlib.Path('out_data') / 'epochs_epo.fif')
epochs_selection = epochs.selection

'''

NÃO ESTOU REFAZENDO AS EPOCHS - Estou basicamente carregando os epochs antigos incluindo todas as exclusões manuais/automáticas feitas anteriormente.

epochs.selection é uma lista de índices que diz:

“Quais epochs originais sobreviveram após rejeições (manuais, automáticas ou via critérios)”.

Ou seja:

- Se você tinha 100 epochs brutas
- Mas rejeitou 10 (clicando na janela interativa ou via threshold)
- Então epochs.selection vai ter 90 números, que são os índices das epochs que sobraram.

epochs = todos os segmentos válidos
epochs.drop_log = motivos e flags de rejeição
epochs.selection = índices dos segmentos aceitos

'''

epochs_selection

Mantenha apenas o subconjunto de eventos que corresponde às epochs retidas.

In [ ]:
events, event_id = mne.events_from_annotations(raw)
events = events[epochs_selection]

events.shape

Crie epochs para o ICA. Todos os parâmetros devem coincidir exatamente com os das epochs que pretendemos limpar (as originais)

In [ ]:
'''
Estamos recriando epochs porque

• Filtramos o raw de novo com o filtro específico para o ICA
• Quero epochs antes do ICA
• Quero epochs depois de aplicar ICA
'''


tmin = -0.3
tmax = 0.5
baseline = (None, 0) # Tempo do começo da epoch até o início do evento

epochs_ica = mne.Epochs(raw,
                        events=events,
                        event_id=event_id,
                        tmin=tmin,
                        tmax=tmax,
                        baseline=baseline,
                        preload=True)

### ICA trabalha com epochs?

**Tericamente não é o ideal, mas o MNE é flexível e permite que você quebre essa regra.**

O MNE é uma biblioteca muito amigável. Quando você passa `epochs` para o `ica.fit()`, ele faz um truque nos bastidores:
1.  Ele pega todos os seus recortes (epochs).
2.  Ele **concatena** (cola) um no final do outro temporariamente.
3.  Ele roda a matemática da ICA nessa "tripa" de dados colados.

Então, para o computador, não deu erro de sintaxe. Ele aceita tanto dados contínuos (`Raw`) quanto recortes (`Epochs`).

#### Por que a teoria diz para usar `Raw` (Contínuo)?
A afirmação que você citou tem três argumentos muito fortes. Vamos analisar cada um:

##### A. "Epocar corta o sinal (saltos na borda)"
Imagine que você tem uma foto panorâmica. Se você cortar ela em vários pedaços e tentar colar de volta, as bordas nunca vão casar perfeitamente. Vai ficar um risco visível.
* **O Problema:** Quando o MNE cola as epochs para calcular a ICA, cria-se um **salto brusco** na emenda (o final da Epoch 1 não liga suavemente com o início da Epoch 2).
* **A ICA odeia isso:** A ICA pode achar que esse "salto" artificial na emenda é um tipo de atividade cerebral e gastar um componente só para tentar consertar essa "cicatriz" de colagem.

##### B. "Epocar já remove trechos com ruído"
Aqui está o paradoxo da ICA: **Para limpar a sujeira, a ICA precisa VER a sujeira.**
* Se, ao criar suas `epochs`, você usou o parâmetro `reject` para jogar fora as épocas onde você piscou o olho...
* ...a ICA nunca vai ver um piscar de olhos durante o treino.
* Conclusão: Ela não vai criar um componente de "piscar de olhos". E se sobrar alguma sujeira sutil, ela não vai saber limpar.

##### C. "Fluxo temporal"
A ICA precisa de estatística. Quanto mais dados contínuos ela tiver, melhor ela entende a diferença entre o que é um batimento cardíaco regular e o que é uma onda cerebral aleatória. Dados contínuos oferecem mais contexto histórico.



#### Quando usar cada um?

**Estratégia Ouro (Recomendada):**
1.  Pega o dado Bruto (`Raw`).
2.  Filtra (1Hz High-pass).
3.  **Roda o `ica.fit` no Raw.** (A ICA aprende com todo o contexto e todas as sujeiras).
4.  Pega essa solução ICA e aplica (`ica.apply`) nas suas Epochs finais.

**Estratégia Prata (O que você fez - Aceitável se tiver cuidado):**
1.  Cria Epochs *sem rejeitar artefatos grosseiros*.
2.  **Roda o `ica.fit` nessas Epochs.**
3.  O MNE concatena tudo e calcula.
    * *Risco:* "Saltos" nas emendas das epochs podem gerar componentes estranhos.
    * *Vantagem:* É mais rápido se o dado Bruto for gigantesco e você só se importa com pequenos pedaços.

### Por fim, ajuste o ICA!

In [ ]:
epochs_ica.info

In [ ]:
# PODE DEMORAR 50 OU > SEGUNDOS (ou mais) para executar

n_components = 0.8  # Should normally be higher, like 0.999!!
method = 'picard'
max_iter = 100  # Should normally be higher, like 500 or even 1000!!
fit_params = dict(fastica_it=5)
random_state = 42

# Cria o objeto ICA com todas as configurações acima
ica = mne.preprocessing.ICA(
    n_components=n_components,        # Quantos componentes manter
    # max_pca_components=300,           Limite máximo de componentes após a PCA inicial ⚠️DEPRECATED⚠️
    method=method,                    # Algoritmo usado
    max_iter=max_iter,                # Iterações permitidas
    fit_params=fit_params,            # Parâmetros extras
    random_state=random_state         # Fixar os resultados
)

'''
Ajusta (treina) o ICA usando seus epochs
Aqui o ICA aprende quais componentes existem no seu dado:
    - componente cardíaco
    - piscada (EOG)
    - movimentos
    - ruído
    - componentes neurais
'''
ica.fit(epochs_ica) # podemos passar epochs ou raw data como argumento


'''
Apareceu um aviso principal

'UserWarning: Picard did not converge)'

    O algoritmo picard tentou separar os sinais, mas o "tempo acabou" antes que ele conseguisse uma separação perfeita.

    Por que aconteceu: As configurações do código forçaram ele a parar cedo demais.

        max_iter = 100 (muito pouco).
        fastica_it=5 dentro de fit_params (extremamente pouco)
'''



### Parâmetros

* `n_components = 0.8`:
    * **O que faz:** Define o quanto de informação (variância) do sinal original você quer manter antes de aplicar a ICA.
    * **Em português claro:** "Mantenha componentes suficientes para explicar 80% do que está acontecendo nos dados".
    * **Nota sobre o comentário:** O comentário diz que deveria ser maior (0.999). Usar 0.8 descarta 20% dos dados como "menos importantes", o que deixa o processo muito mais rápido, mas pode jogar fora detalhes sutis do cérebro.

* `method = 'picard'`:
    * **O que faz:** Escolhe o algoritmo matemático ("a receita") que será usado para separar os sinais.
    * **Em português claro:** Existem vários métodos (FastICA, Infomax). O `picard` é uma variação moderna e eficiente do FastICA.
    * 'picard' é rápido e funciona muito bem para EEG.

* `max_iter = 100`:
    * **O que faz:** É o número máximo de tentativas que o algoritmo fará para separar os sinais perfeitamente.
    * **Em português claro:** O computador vai tentar ajustar a separação até 100 vezes. Se ele não conseguir um resultado perfeito até lá, ele para e te entrega o melhor que conseguiu. (Normalmente usa-se 500 ou 1000 para garantir uma separação de alta qualidade).

* `fit_params = dict(fastica_it=5)`:
    * **O que faz:** São ajustes finos específicos para o método escolhido (`picard`).
    * **Em português claro:** Aqui você está dizendo para a parte interna do algoritmo rodar apenas 5 iterações. Isso é **extremamente baixo**. Serve apenas para testar se o código roda sem erros, pois o resultado da limpeza provavelmente será ruim com esse valor.

* `random_state = 42` (opicional):
    * **O que faz:** É a "semente" da aleatoriedade.
    * **Em português claro:** Garante que, toda vez que você rodar esse código, o resultado seja **exatamente o mesmo**. Sem isso, cada vez que você rodasse, os componentes poderiam sair numa ordem ligeiramente diferente.



### Criando o Objeto e Rodando

* `ica = mne.preprocessing.ICA(...)`:
    * **O que faz:** Aqui você cria a "máquina" de ICA com todas as regras que definiu acima. Ela ainda está vazia, apenas configurada.
    * *Nota:* O `max_pca_components=300` é um teto de segurança para não explodir a memória do computador antes de reduzir para o `n_components`.

* `ica.fit(epochs_ica)`:
    * **O que faz:** É aqui que a mágica acontece. O comando `.fit()` pega os seus dados de EEG (`epochs_ica`) e "treina" a ICA neles.
    * **Resultado:** Depois dessa linha, a variável `ica` terá aprendido como separar os olhos, o coração e o cérebro dentro dos seus dados.

*Obs:* ICA() não aceita mais o argumento max_pca_components. Esse parâmetro existia em versões antigas do MNE, mas foi removido nas versões atuais.

**E o PCA então? Como controla?**

No MNE atual:

- n_components controla tudo.
- Se você coloca um float (ex: 0.8 ou 0.999), isso significa:
- "explique X% da variância da PCA".

In [ ]:
#   28 componentes recuperados! Entre eles vamos achar nossos artefatos
ica.plot_components(inst=epochs_ica) # um plot em 2 janelas (para caber os 28) e interativo. Se você clicar ele detalha cada componente (demora para abrir mesmo)

## Components

### Por que 28?

O algoritmo pegou todos os seus canais (digamos que você tenha 64 eletrodos) e condensou a informação. Ele calculou que **28 "fontes" de sinal** são suficientes para explicar aquela porcentagem dos seus dados.
* Se você tivesse colocado `n_components = 28`, ele teria buscado exatamente 28.
* Como você colocou uma porcentagem (0.8), ele calculou sozinho que precisava de 28 componentes.

### O que são as "Imagens de Cabeça" (Topomaps)?
Essas "cabeças" são **mapas de calor** chamados *Topomaps*. Elas não mostram atividade cerebral direta, elas mostram **PESO e LOCALIZAÇÃO**.

* **Vermelho/Azul forte:** Significa que aquele componente aparece muito forte nesses eletrodos.
* **Verde/Branco:** Significa que aquele componente quase não afeta essa região.

**O segredo para ler:**
* Se você vê uma mancha vermelha bem em cima dos olhos (na frente da cabeça), esse componente é quase certeza o seu **piscar de olhos**.
* Se você vê manchas vermelhas isoladas nas têmporas, provavelmente é **tensão muscular** (morder).

### O que é um Componente ICA? (A analogia da Banda)

Imagine que você gravou uma banda tocando com 64 microfones espalhados pelo estúdio (seus 64 eletrodos).
* **O problema:** Cada microfone gravou tudo misturado (bateria + guitarra + voz + barulho do ar condicionado).
* **O objetivo:** Você quer ouvir só a guitarra.

O **Componente ICA** é a tentativa matemática de separar cada instrumento individualmente.
* **Componente 01:** É a faixa de áudio só da bateria.
* **Componente 02:** É a faixa de áudio só da guitarra.
* **Componente 03:** É a faixa de áudio só do ar condicionado (RUÍDO!).

No seu EEG:
* Um componente é o **"Sinal do Piscar de Olhos"** isolado.
* Outro componente é o **"Batimento Cardíaco"** isolado.
* Outro componente é a **"Onda Alfa"** isolada.



### Como ele vai usar isso? (O processo de limpeza)
Agora que você tem as 28 "faixas de áudio" separadas, o processo de limpeza é manual ou automático:

1.  **Identificação:** Você (ou um algoritmo) olha para as 28 cabecinhas e para os gráficos de onda delas.
2.  **Julgamento:** Você diz: *"Opa, o Componente 0 e o Componente 4 têm cara de piscar de olhos e batimento cardíaco. Eles são lixo."*
3.  **Exclusão:** Você marca esses componentes para exclusão (`ica.exclude = [0, 4]`).
4.  **Reconstrução:** O computador pega os 26 componentes restantes (os bons) e **remixa a música**.

**O Resultado Final:** Você tem seus dados de volta nos canais originais, mas agora a "faixa de áudio" do piscar de olhos foi removida magicamente, sem estragar o resto dos dados.

### Detect ECG and EOG patterns

In [ ]:
# Vamos deixar o prórpio MNE indentificar os artefatos

# Ele procura picos no canal de ECG (ou seja, >BATIDAS DO CORAÇÃO<) e corta janelas de -0.5s a +0.5s ao redor de cada batida
ecg_epochs = mne.preprocessing.create_ecg_epochs(raw, reject=None,
                                                 baseline=(None, -0.2),
                                                 tmin=-0.5, tmax=0.5)

# Tira a média desses recortes para visualizar o "formato padrão" do artefato cardíaco
# Isso cria o perfil do "inimigo" que queremos remover
ecg_evoked = ecg_epochs.average()

# O algoritmo compara os componentes da ICA com os batimentos cardíacos
# ecg_inds: Lista com os números dos componentes ruins (ex: [0, 15])
# ecg_scores: Notas de "culpa" (quão parecido o componente é com o coração)- 
# É um valor matemático (geralmente variando entre 0 e 1) que diz o quão forte é a conexão entre aquele componente e o batimento cardíaco.
ecg_inds, ecg_scores = ica.find_bads_ecg(
    ecg_epochs, method='ctps') # 'ctps' é um método estatístico robusto para isso


# Cria "recortes" focados apenas nas piscadas de olho
# Ele procura grandes saltos de energia nos canais frontais ou canal EOG
eog_epochs = mne.preprocessing.create_eog_epochs(raw, reject=None,
                                                 baseline=(None, -0.2),
                                                 tmin=-0.5, tmax=0.5)
# Tira a média para ver o "formato padrão" da piscada
eog_evoked = eog_epochs.average()

# Caça aos componentes culpados (Detecção Automática)
# Compara os componentes ICA com as piscadas para ver quais são idênticos
eog_inds, eog_scores = ica.find_bads_eog(
    eog_epochs)

# Atualiza a "Lista Negra" (ica.exclude é uma lista) da ICA
# Importante: Isso NÃO remove nada ainda, apenas marca os componentes para
# serem removidos quando você aplicar a ICA (.apply()) mais tarde.
components_to_exclude = ecg_inds + eog_inds
ica.exclude = components_to_exclude



### O objeto ICA



####  Ele armazena os componentes ou é só visualização?

**Ela armazena a MATEMÁTICA para achar os componentes.**

Pense no objeto `ica` como um **Óculos de Filtro Especial** (ou um Prisma).

  * Ele **não** guarda a luz (os dados brutos).
  * Ele guarda a **receita da lente** (a matriz de pesos matemáticos).

Quando você roda `ica.plot_components()`, o objeto pega essa receita matemática, aplica rapidinho nos dados e te mostra o desenho. Se você deletar os dados brutos da memória, o objeto `ica` continua existindo com a "receita", mas não consegue mais desenhar nada porque não tem dados para processar.


#### O que el armazena de fato

Ele é leve. Ele armazena principalmente três coisas:

1.  **A Matriz de Mistura (`mixing_matrix_`):** A fórmula matemática que explica como os 28 componentes se misturam para criar seus canais de MEG.
2.  **A Matriz de Desmistura (`unmixing_matrix_`):** A fórmula inversa (o liquidificador reverso).
3.  **A Lista Negra (`ica.exclude`):** Uma lista simples de números (ex: `[0, 15]`) que diz quais componentes são "persona non grata".

**Resumo:** O objeto `ica` é o **mapa do tesouro**. <u>Ele não é o tesouro (dados)</u>, mas ele sabe exatamente onde cavar para separar o ouro (sinal) da terra (ruído).


#### Ele vai propor mudanças ou só vai me mostrar os componentes?

Até agora,ele é **PASSIVO**.

1.  **`ica.fit`**: Ele aprendeu a separar. (Passivo)
2.  **`ica.find_bads_...`**: Ele **sugeriu** quem são os culpados e colocou na lista `ica.exclude`. (Proposta)
3.  **A Mágica Real (`ica.apply`):** Você **ainda não rodou** esse comando.

O objeto `ica` funciona assim:

  * Ele segura os dados na mão esquerda.
  * Segura a lista de exclusão (`[0, 15]`) na mão direita.
  * Ele **só vai alterar** os seus dados quando você digitar:
    ```python
    epochs_limpas = ica.apply(epochs_ica)
    ```
    Até lá, seus dados continuam sujos. O `ica` está apenas "preparado" para limpar.




In [ ]:
ica.exclude

## Plotar 'scores' de detecção automática de artefatos


Pense nesse "score" não como uma "nota de qualidade" (como numa prova), mas sim como um "Nível de Contaminação" ou uma "Porcentagem de Plágio".
```markdown
    >Score Alto< (ex: 0.90): Significa PERIGO/RUÍDO. O componente é 90% idêntico ao batimento cardíaco. Ele é "lixo" e deve ser removido.
        - Score Negativo (-0.9) É RUIM TBM. Para o computador, os dois são o coração.
        - Se o ecg_score for muito negativo, isso significa que aquele componente é uma cópia invertida do seu batimento cardíaco.
    >Score Baixo< (ex: 0.01): Significa BOM/CÉREBRO. O componente não tem nada a ver com o coração. Ele é o sinal que você quer manter.
```

- Perto de +1: Culpado (Cópia fiel).
- Perto de -1: Culpado (Cópia invertida).
- Perto de 0: Inocente (Não tem relação).

**Fiquei com muita dúvida sobre o fato de estarmos removendo um sinal que, em tese, é o princípio do ECG, mas, percebi que ele é só um coadjuvante**

Você **não é um Cardiologista**, você é um **Neurocientista**.

1.  **Se você fosse Cardiologista:** Você iria querer **manter** o sinal do coração para ver se o paciente está infartando.
2.  **Como Neurocientista (seu caso agora):** O sinal do coração é **LIXO**. Ele é uma interferência elétrica que "suja" a sua leitura do cérebro.

### O Problema: O "Vazamento" Elétrico
O coração é um músculo elétrico muito forte. A eletricidade dele é tão forte que viaja pelo pescoço e aparece nos eletrodos que estão na cabeça.


#### O Processo (O "Molde")
O computador não sabe o que é coração e o que é cérebro. Ele só vê números.

Ao passar o `ecg_epochs` (o sinal puro do coração) para ele, você está dizendo:
> *"Computador, olhe para este sinal aqui (ECG). Este é o formato do inimigo. Agora, vá lá nos meus dados de cérebro (EEG) e veja se alguém tem o desenho igualzinho a esse."*

#### A Analogia da Tinta
Imagine que caiu tinta vermelha (coração) na sua pintura azul (cérebro).
* Você tem um **pote de tinta vermelha pura** (o canal ECG) ao seu lado.
* Você mostra esse pote para o computador e diz: *"Computador, tudo o que tiver **esta cor exata** na minha pintura, apague."*



### Resumo
Você usa o canal de ECG **não porque quer analisar o coração, mas porque precisa dele como *Gabarito (Referência)*. Sem o canal de ECG puro, o computador teria que "adivinhar" o que é batimento cardíaco. Com o canal de ECG, ele tem certeza absoluta.**

Por isso seus comentários estão corretos:
1.  Você pega o ECG.
2.  Tira a média para ver o "perfil do inimigo".
3.  Usa esse perfil para caçar os infiltrados no cérebro.

    

![Correação Scores](imgs/correlation-scores-ICA.jpeg)

In [ ]:
ica.plot_scores(ecg_scores) # vamos visualizar o score que ele dá para cada dado


In [ ]:
ica.plot_scores(eog_scores) # completamente invertido, o componente 000 é muito negativo, então será lido como ruído assosciado a EOG (movimento dos olhos)

## Plotar ICA sources

In [ ]:
ica.plot_sources(ecg_evoked) # Visualização bonita e bem informativa

'''
plot_sources é um comando do MNE que mostra a atividade de cada componente ICA ao longo do tempo.
Ele precisa de algum objeto MNE que contenha um sinal no mesmo formato dos dados usados na ICA.

- Esse objeto pode ser:
- raw
- epochs
- evoked

O ecg_evoked é a média dos batimentos cardíacos detectados (ECG).
Mas o plot não é sobre o ecg_evoked em si.
Plota os componentes ICA usando o ecg_evoked como sinal de entrada.

“Como cada componente se ativa quando ocorre um batimento cardíaco?””

Cada linha colorida ou preta é um Componente ICA

- Eixo X (Tempo): O centro (0 ms) é o momento exato do pico do batimento cardíaco.
- Linhas Pretas (A multidão inocente): A maioria das linhas está "bagunçada" ou reta perto do zero. Isso significa que, quando o coração bate, esses componentes não reagem. Eles são sinais cerebrais (ou outros ruídos) que não têm nada a ver com o coração.
- Linhas Coloridas e com Picos (Os culpados): Olhe para a linha Azul (ICA000) e a linha Ciano (ICA014). Elas dão um salto gigante exatamente no tempo 0.

    Isso prova visualmente que esses componentes SÃO o batimento cardíaco. Eles estão sincronizados perfeitamente com o ECG.
    
    Note que a linha marrom não explode nesse gráfico. Por que? Porque esse gráfico mostra a reação ao coração. 
    Como o olho não pisca no ritmo do coração, a linha do olho fica quieta aqui. 
    Se você plotasse o eog_evoked, essa linha marrom seria a que daria um salto gigante.
    
    
- O eixo Y representa a Amplitude (a força ou intensidade) do sinal daquele componente, mas com um detalhe importante: ele não tem unidade física.

Se você olhar bem no canto esquerdo da sua imagem, vai ver escrito (NA). Isso significa "Not Available" (Não Disponível) ou "No Unit" (Sem Unidade).
'''


### `ica.plot_sources(ecg_evoked)`: Por que plotou tudo se passei só o ECG?

Essa é a confusão clássica entre **Canal (Sensor)** e **Componente (Fonte)**.

Quando você passa o `ecg_evoked` para o ICA, você está dizendo:

> *"Tome aqui esse pedacinho de tempo (o momento da batida do coração). Agora, passe esse pedacinho pelo seu 'Prisma Mágico' e me mostre no que ele se decompõe."*

> O ecg_evoked é, na verdade, a gravação de TODOS os seus sensores (MEG/EEG) no exato momento em que o coração bateu.


  * **O Input (`ecg_evoked`):** É a "luz branca" (o sinal misturado registrado pelos sensores).
  * **O Processo:** O ICA passa isso pela matriz matemática.
  * **O Output (O gráfico):** É o "arco-íris" (os 28 componentes).

O gráfico está mostrando:

  * *"Durante a batida do coração, o componente 0 gritou alto."* (Linha azul explodindo).
  * *"Durante a batida do coração, o componente 5 ficou quieto."* (Linha preta reta).
  * *"Durante a batida do coração, o componente 27 ficou quieto."* (Linha preta reta).

Ele tem que mostrar todos os 28 para provar para você que **apenas** os componentes identificados (azul/ciano) reagiram ao coração, enquanto os outros (cérebro) ignoraram.

## Plote a sobreposição (overlay) dos dados originais e limpos

In [ ]:
ica.plot_overlay(ecg_evoked) # Mostra como o dado parece depois de remover os daa lista 'ica.exclude'

# Ele diz: "Olha, se você confirmar essa exclusão, é assim que o seu sinal vai ficar."

'''
Linha Limpa (Geralmente Preta): Mostra como os seus dados ficariam se você removesse os componentes que marcou para exclusão.
Linha Original (Geralmente Vermelha): Mostra os dados sujos, como o batimento cardíaco nesse caso.
'''

<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
    <ul>
        <li>Visualize os escores de detecção de artefatos, as fontes ICA e sobreponha os dados originais e limpos com base nas épocas de EOG.</li>
        <li>Visualize os escores de detecção de artefatos, as fontes ICA e sobreponha os dados originais e limpos com base nas épocas que realmente pretendemos analisar.</li>
    </ul>
</div>


In [ ]:
# ---Resposta --- #

# Scores: O gráfico de barras dos culpados
# Mostra quais componentes têm alta correlação com o canal EOG
ica.plot_scores(eog_scores)

In [ ]:
# ---Resposta --- #

ica.plot_sources(eog_evoked) 

In [ ]:
# ---Resposta --- #

ica.plot_overlay(eog_evoked)

In [ ]:
# ---Resposta --- #
# Não plotamos "scores" aqui de novo porque os scores são calculados contra o EOG/ECG, não contra suas epochs de experimento.
ica.plot_scores(epochs_ica)

In [ ]:
# ---Resposta --- #

# Sources: Olhando os componentes "vivos" nas suas epochs
# Isso abre uma janela interativa onde você pode ver os componentes ao longo de todo o seu experimento.
# Útil para ver se sobrou algum ruído que o automático não pegou.
ica.plot_sources(epochs) 

In [ ]:
# ---Resposta --- #

# Overlay: O teste do "Antes vs Depois" no seu experimento
# IMPORTANTE: O ICA precisa de um objeto 'Evoked' para fazer o overlay (uma média).
# Então usamos .average() nas suas epochs_ica.
# O QUE ESPERAR:
# - Se a linha vermelha for IGUAL à preta: Você não limpou nada.
# - Se a linha vermelha for MUITO diferente em todo lugar: Você removeu cérebro demais.
# - O ideal: Diferenças pontuais (onde havia ruído) e preservação dos picos principais do cérebro
ica.plot_overlay(epochs.average())

## Excluindo componentes relacionados a artefatos

- Aplicamos a exclusão aos epochs que produzimos anteriormente (o que de fatos que realmente queremos limpar para analisar)
- O epochs_ica é só para detecção dos artefatos de EOG e ECG com máxima precisão

In [ ]:
epochs_cleaned = ica.apply(epochs.copy()) # in-place também

In [ ]:
epochs_cleaned.plot()

In [ ]:
# Compare com a versão anterior

epochs.plot()

In [ ]:
epochs_cleaned.save(pathlib.Path('out_data') / 'epochs_cleaned_epo.fif', overwrite=True) # no final, são epochs

É ótimo a automatização para remoção de artefatos EOG e ECG, mas, para outros mais complexos isso deve ser manual